# 03 - Ablations: which part is doing the work?

A method that only reports its final number is not testable. Two studies:

**Component ablation** - disable one mechanism at a time, so each drop is
attributable to that mechanism and not to the bundle.

**Labelled-fraction sweep** - re-train at 5/10/20/50% labels with the gradient
step count held fixed. Any semi-supervised method looks good at 50% labels,
where the supervised branch dominates; the question is the 5-10% regime. If the
gain does not shrink as labels are added, the method is probably acting as a
regulariser rather than exploiting the unlabelled pool.

Two design details make the sweep a controlled experiment rather than a set of
unrelated draws:

- `optim.steps_per_epoch` is **fixed**, so training length does not vary with
  label count.
- Labelled subsets are **nested** - raising the fraction strictly adds images.

In [ ]:
# Run from the repository root, or from notebooks/ - both work.
import sys, os
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
    os.chdir(root)
sys.path.insert(0, str(root / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

torch.set_num_threads(max(1, (os.cpu_count() or 2)))
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print("repo root :", root)
print("torch     :", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
QUICK = True   # the sweep re-trains many times; start small

SCALE = ([
    "data.image_size=48", "data.train_size=200", "data.val_size=32",
    "data.test_size=64", "data.batch_size=6", "data.mu=1",
    "model.width=12", "model.depth=3",
    "optim.epochs=6", "optim.steps_per_epoch=8",
    "loss.kl_anneal_epochs=3", "semi.rampup_epochs=2", "eval.bootstrap=500",
] if QUICK else [
    "data.image_size=64", "data.train_size=600", "data.val_size=80",
    "data.test_size=150", "data.batch_size=8", "data.mu=2",
    "model.width=16", "model.depth=3",
    "optim.epochs=24", "optim.steps_per_epoch=24",
    "loss.kl_anneal_epochs=10", "semi.rampup_epochs=8",
])

## Nested labelled subsets, verified

The property the sweep depends on, checked before spending compute on it.

In [ ]:
import dataclasses
from evissl.config import DataConfig
from evissl.data import build_dataset

base = DataConfig(name="synthetic", image_size=32, train_size=400,
                  val_size=8, test_size=8)
previous, rows = set(), []
for fraction in (0.05, 0.10, 0.20, 0.50, 1.0):
    bundle = build_dataset(dataclasses.replace(base, labeled_fraction=fraction), seed=1337)
    current = set(bundle.labeled_idx.tolist())
    rows.append({
        "fraction": fraction,
        "n_labeled": len(current),
        "is superset of previous": previous.issubset(current),
        "added": len(current - previous),
    })
    previous = current
display(pd.DataFrame(rows))

## Component ablation

In [ ]:
from evissl.pipelines import run_component_ablation

ablation = run_component_ablation("configs/evidential.yaml", overrides=SCALE)
display(ablation.round(4))

In [ ]:
if not ablation.empty and "delta_dice" in ablation:
    view = ablation[ablation["variant"] != "full"].sort_values("delta_dice")
    fig, ax = plt.subplots(figsize=(7.5, 0.55 * len(view) + 1.6))
    ax.barh(view["variant"], view["delta_dice"], color="#009E73")
    ax.axvline(0, color="black", lw=1)
    ax.set_xlabel("change in Dice when this component is removed")
    ax.set_title("Component ablation (negative = the component was helping)")
    ax.grid(axis="y", visible=False)
    plt.tight_layout(); plt.show()

What each row tests:

| variant | removes | question it answers |
|---|---|---|
| `no_dissonance_temper` | per-pixel temperature | is softening ambiguous boundaries worth anything? |
| `no_vacuity_gate` | belief-mass weighting | is the weighting doing the work, or just the loss? |
| `no_kl` | evidential KL regulariser | does vacuity stay informative without it? |
| `no_axial` | bottleneck attention | how much is architecture rather than ideology? |
| `gate_without_evidential_loss` | the evidential *objective*, keeping the gate | does the rule need the training objective, or just the arithmetic? |

`no_kl` is the one to watch. Without a penalty on misleading evidence the
network can lower its loss by inflating evidence everywhere; vacuity then
collapses towards zero for every pixel and the gate degenerates into Mean
Teacher.

## Labelled-fraction sweep

In [ ]:
from evissl.pipelines import run_labeled_fraction_sweep
from evissl.report import write_sweep_figure

sweep = run_labeled_fraction_sweep(
    ["configs/supervised_baseline.yaml", "configs/fixmatch.yaml",
     "configs/evidential.yaml"],
    fractions=(0.05, 0.10, 0.20, 0.50),
    overrides=SCALE,
)
display(sweep["table"].round(4))
print(write_sweep_figure(sweep, "results"))

In [ ]:
from evissl.viz import plot_sweep

fig = plot_sweep(sweep["sweep"], metric="dice",
                 xlabel="labelled fraction of the training set",
                 title="Where does the semi-supervised gain live?")
plt.show()

# The gain over the supervised lower bound, per label budget.
frame = sweep["table"]
if {"method", "labeled_fraction", "dice"} <= set(frame.columns):
    wide = frame.pivot_table(index="labeled_fraction", columns="method", values="dice")
    if "supervised" in wide:
        gain = wide.drop(columns=["supervised"]).sub(wide["supervised"], axis=0)
        print("\nDice gain over the supervised lower bound:")
        display(gain.round(4))

Read the gain table, not the absolute numbers. A healthy semi-supervised method
shows its largest gain at the smallest label budget and converges towards the
supervised bound as labels accumulate - because there is progressively less for
the unlabelled pool to add.